# 🫘 Dry Bean Classifier
**Question:** Can we tell seven varieties apart — for every variety, not just the common ones?

In [ ]:
# Choose the data mode: "sample" runs offline on the synthetic fixture,
# "real" downloads/queries the official source the first time and caches it in data/raw/.
MODE = "sample"
OUT = f"outputs/notebook-{MODE}"

import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
import load_data, analysis

Path(OUT).mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)
MODEL = 'logreg'  # or 'random_forest'

> **Sample mode** uses invented data with the real schema. Every chart is stamped *SYNTHETIC SAMPLE*. Change `MODE = "real"` above to run on the official data.

## 1 · Load and validate

In [ ]:
df = load_data.load(MODE)
print(df.shape)
df[load_data.TARGET].value_counts()

## 2 · Class balance

In [ ]:
display(Image(f"{OUT}/{analysis.plot_class_balance(df, Path(OUT), MODE == 'sample')}"))

## 3 · Stratified split
Without stratification a rare class can vanish from the test set.

In [ ]:
X_tr, X_te, y_tr, y_te = analysis.split(df, 0.25)
pd.DataFrame({'train': y_tr.value_counts(), 'test': y_te.value_counts()})

## 4 · Baseline: always predict the majority class

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
dummy = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
yb = dummy.predict(X_te)
print('accuracy', round(accuracy_score(y_te, yb), 3), '| macro F1', round(f1_score(y_te, yb, average='macro'), 3))

## 5 · Classifier inside a pipeline
The scaler is part of the pipeline, so it is fit on training data only.

In [ ]:
model = analysis.make_model(MODEL).fit(X_tr, y_tr)
ym = model.predict(X_te)
print('accuracy', round(accuracy_score(y_te, ym), 3), '| macro F1', round(f1_score(y_te, ym, average='macro'), 3))

## 6 · Per-class results

In [ ]:
classes = sorted(df[load_data.TARGET].unique())
tbl = analysis.evaluate(y_te, ym, classes)
display(tbl.round(3))
display(Image(f"{OUT}/{analysis.plot_confusion(y_te, ym, classes, Path(OUT), MODE == 'sample')}"))
display(Image(f"{OUT}/{analysis.plot_per_class(analysis.evaluate(y_te, yb, classes), tbl, Path(OUT), MODE == 'sample')}"))

## 7 · Limitations
- One imaging setup; features only.
- Correlated features make importances hard to interpret.

## Next
Separate SIRA from DERMASON, or compare models with stratified 5-fold CV.